Import libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler

In [2]:
df = pd.read_csv("../data/raw/creditcard.csv")
df = df.sort_values("Time").reset_index(drop=True)

print(df.shape)
df.head()

(284807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
df_fe = df.copy()

TRANSFORMATION FEATURES

In [4]:
df_fe["Log_Amount"] = np.log1p(df_fe["Amount"])

In [5]:
robust_scaler = RobustScaler()
df_fe["Amount_Robust"] = robust_scaler.fit_transform(df_fe[["Amount"]])

TEMPORAL FEATURES

In [6]:
df_fe["Hour"] = (df_fe["Time"] // 3600) % 24

In [7]:
df_fe["Day"] = df_fe["Time"] // (24 * 3600)

In [8]:
df_fe["time_diff"] = df_fe["Time"].diff().fillna(0)

ROLLING BEHAVIORAL FEATURES

In [9]:
df_fe["amount_roll_mean_5"] = df_fe["Amount"].rolling(window=5, min_periods=1).mean()
df_fe["amount_roll_mean_10"] = df_fe["Amount"].rolling(window=10, min_periods=1).mean()

In [10]:
df_fe["amount_roll_std_5"] = df_fe["Amount"].rolling(window=5, min_periods=1).std().fillna(0)
df_fe["amount_roll_std_10"] = df_fe["Amount"].rolling(window=10, min_periods=1).std().fillna(0)

In [11]:
df_fe["amount_ratio_5"] = df_fe["Amount"] / (df_fe["amount_roll_mean_5"] + 1e-6)
df_fe["amount_ratio_10"] = df_fe["Amount"] / (df_fe["amount_roll_mean_10"] + 1e-6)

In [12]:
df_fe["amount_roll_median_10"] = df_fe["Amount"].rolling(window=10, min_periods=1).median()

In [13]:
df_fe["amount_dev_median_10"] = df_fe["Amount"] - df_fe["amount_roll_median_10"]

In [14]:
df_fe["tx_count_proxy_5"] = df_fe["Time"].rolling(window=5, min_periods=1).count()
df_fe["tx_count_proxy_10"] = df_fe["Time"].rolling(window=10, min_periods=1).count()

TƯƠNG TÁC GIỮA FEATURE GỐC VÀ FEATURE MỚI

In [15]:
df_fe["amount_time_interaction"] = df_fe["Amount"] / (df_fe["time_diff"] + 1)

In [16]:
df_fe["log_amount_hour_interaction"] = df_fe["Log_Amount"] * df_fe["Hour"]

KIỂM TRA FEATURE SAU KHI TẠO

In [17]:
df_fe.isnull().sum().sort_values(ascending=False).head(20)
print("NaN count:", df_fe.isnull().sum().sum())
print("Inf count:", np.isinf(df_fe.select_dtypes(include=[np.number])).sum().sum())

NaN count: 0
Inf count: 0


In [18]:
print("Original shape:", df.shape)
print("Feature-engineered shape:", df_fe.shape)

Original shape: (284807, 31)
Feature-engineered shape: (284807, 48)


In [19]:
corr_with_target = df_fe.corr(numeric_only=True)["Class"].sort_values(ascending=False)
print(corr_with_target.head(15))
print(corr_with_target.tail(15))

Class                   1.000000
V11                     0.154876
V4                      0.133447
V2                      0.091289
V21                     0.040413
V19                     0.034783
V20                     0.020090
V8                      0.019875
time_diff               0.018341
V27                     0.017580
V28                     0.009536
amount_ratio_10         0.006294
amount_dev_median_10    0.005811
Amount                  0.005632
Amount_Robust           0.005632
Name: Class, dtype: float64
Time                          -0.012323
log_amount_hour_interaction   -0.013905
Hour                          -0.017109
V6                            -0.043643
V5                            -0.094974
V9                            -0.097733
V1                            -0.101347
V18                           -0.111485
V7                            -0.187257
V3                            -0.192961
V16                           -0.196539
V10                           -0.2168

FEATURE SELECTION


In [20]:
features_to_drop = []
df_fe = df_fe.drop(columns=features_to_drop)

CHUẨN BỊ DATASET

In [21]:
X = df_fe.drop(columns=["Class"])
y = df_fe["Class"]

In [22]:
split_idx = int(len(df_fe) * 0.8)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(227845, 47) (56962, 47)
0.001830191577607584 0.001316667251852112


In [23]:
train_df = X_train.copy()
train_df["Class"] = y_train.values

test_df = X_test.copy()
test_df["Class"] = y_test.values

train_df.to_csv("../data/processed/train_fe.csv", index=False)
test_df.to_csv("../data/processed/test_fe.csv", index=False)

In the feature engineering stage, the original transaction-level dataset was transformed into a richer representation that captures temporal and behavioral characteristics of fraud. Since the dataset does not provide explicit customer or card identifiers, user-level aggregation techniques commonly used in industrial fraud systems could not be directly applied. Instead, sequential behavioral features were constructed from transaction order and timing, including inter-transaction time gaps, rolling transaction amount statistics, deviation from recent transaction patterns, and time-based indicators such as hour of day and day index. In addition, the transaction amount was log-transformed and robust-scaled to reduce skewness and mitigate the influence of extreme outliers. These engineered features were designed to improve the model’s ability to detect abnormal transaction behavior rather than relying solely on the anonymized PCA-based variables.

In [24]:
df_fe.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,amount_roll_std_5,amount_roll_std_10,amount_ratio_5,amount_ratio_10,amount_roll_median_10,amount_dev_median_10,tx_count_proxy_5,tx_count_proxy_10,amount_time_interaction,log_amount_hour_interaction
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.000000,0.000000,1.000000,1.000000,149.620,0.000,1.0,1.0,149.620,0.0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,103.895199,103.895199,0.035323,0.035323,76.155,-73.465,2.0,2.0,2.690,0.0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,189.473475,189.473475,2.139443,2.139443,149.620,229.040,3.0,3.0,189.330,0.0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,156.999237,156.999237,0.754809,0.754809,136.560,-13.060,4.0,4.0,123.500,0.0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,142.266623,142.266623,0.483049,0.483049,123.500,-53.510,5.0,5.0,34.995,0.0


In [25]:
feature_list = X_train.columns.tolist()
print(feature_list)
print("Number of features:", len(feature_list))

['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Log_Amount', 'Amount_Robust', 'Hour', 'Day', 'time_diff', 'amount_roll_mean_5', 'amount_roll_mean_10', 'amount_roll_std_5', 'amount_roll_std_10', 'amount_ratio_5', 'amount_ratio_10', 'amount_roll_median_10', 'amount_dev_median_10', 'tx_count_proxy_5', 'tx_count_proxy_10', 'amount_time_interaction', 'log_amount_hour_interaction']
Number of features: 47


 Time-Based Train-Test Split Diagram




In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import FancyArrowPatch, Rectangle

# Figure 3.5: Time-based train-test split diagram
# If the split variables already exist, this cell uses them.
# Otherwise, it loads the saved feature-engineered train/test files.

try:
    X_train
    X_test
    y_train
    y_test
except NameError:
    train_df = pd.read_csv("../data/processed/train_fe.csv")
    test_df = pd.read_csv("../data/processed/test_fe.csv")

    X_train = train_df.drop(columns=["Class"])
    y_train = train_df["Class"]
    X_test = test_df.drop(columns=["Class"])
    y_test = test_df["Class"]

figure_dir = Path("../reports/thesis_figures")
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / "figure_3_5_time_based_train_test_split.png"

train_rows = len(X_train)
test_rows = len(X_test)
total_rows = train_rows + test_rows
train_ratio = train_rows / total_rows
test_ratio = test_rows / total_rows
train_fraud_ratio = y_train.mean()
test_fraud_ratio = y_test.mean()
feature_count = X_train.shape[1]

fig, ax = plt.subplots(figsize=(10.5, 3.6))
ax.set_xlim(0, 100)
ax.set_ylim(0, 1)
ax.axis("off")

train_color = "#2E86AB"
test_color = "#F28E2B"
boundary_color = "#333333"

left_margin = 4
right_margin = 96
usable_width = right_margin - left_margin
train_width = usable_width * train_ratio
test_width = usable_width * test_ratio
split_x = left_margin + train_width

ax.add_patch(
    Rectangle(
        (left_margin, 0.38),
        train_width,
        0.25,
        facecolor=train_color,
        edgecolor="black",
        linewidth=1.2,
    )
)
ax.add_patch(
    Rectangle(
        (split_x, 0.38),
        test_width,
        0.25,
        facecolor=test_color,
        edgecolor="black",
        linewidth=1.2,
    )
)

ax.plot(
    [split_x, split_x],
    [0.27, 0.75],
    color=boundary_color,
    linestyle="--",
    linewidth=1.4,
)

ax.text(
    left_margin + train_width / 2,
    0.505,
    (
        "Training + validation set\n"
        "Earlier transactions\n"
        f"{train_rows:,} rows ({train_ratio:.0%})\n"
        f"{feature_count} features\n"
        f"Fraud ratio: {train_fraud_ratio:.6f}"
    ),
    ha="center",
    va="center",
    color="white",
    fontsize=10,
    fontweight="bold",
)
ax.text(
    split_x + test_width / 2,
    0.505,
    (
        "Final test set\n"
        "Later transactions\n"
        f"{test_rows:,} rows ({test_ratio:.0%})\n"
        f"{feature_count} features\n"
        f"Fraud ratio: {test_fraud_ratio:.6f}"
    ),
    ha="center",
    va="center",
    color="white",
    fontsize=9,
    fontweight="bold",
)

ax.text(
    50,
    0.86,
    "Chronological transaction order based on Time",
    ha="center",
    va="center",
    fontsize=13,
    fontweight="bold",
)

ax.add_patch(
    FancyArrowPatch(
        (8, 0.78),
        (93, 0.78),
        arrowstyle="->",
        mutation_scale=16,
        linewidth=1.8,
        color=boundary_color,
    )
)
ax.text(8, 0.72, "Older transactions", ha="left", va="center", fontsize=9)
ax.text(93, 0.72, "Newer transactions", ha="right", va="center", fontsize=9)

ax.text(
    split_x,
    0.23,
    "Split boundary",
    ha="center",
    va="center",
    fontsize=9,
    color=boundary_color,
    fontweight="bold",
)

ax.text(
    50,
    0.13,
    (
        "No random shuffle: the model learns from earlier transactions "
        "and is evaluated on future unseen transactions."
    ),
    ha="center",
    va="center",
    fontsize=9.5,
    color="#333333",
)

plt.tight_layout()
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Figure 3.5 saved to: {figure_path}")
